# Guided Waves: from two wires to a fibre

Companion to `antenna_propagation.ipynb`. There the structure was designed to *lose* energy to free space; here it is designed to *keep* it. One question runs through all six sections: what must a structure do to hold a wave bound to itself?

The answer changes three times. A two-conductor line guides any frequency down to DC and needs no cutoff. A hollow metal pipe has no second conductor, so it can only support patterns that fit across its width — discrete modes, each dead below its own cutoff frequency. A dielectric rod has no conductor at all and traps light by total internal reflection instead. Each step removes a conductor and something must replace it.

$$\text{TEM line (2 conductors)}\ \longrightarrow\ \text{metal guide (1 conductor)}\ \longrightarrow\ \text{dielectric guide (0 conductors)}$$

In [1]:
%matplotlib inline
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display
from scipy.optimize import brentq

plt.rcParams.update({
    "figure.dpi": 108,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.3,
    "font.size": 9,
    "axes.titlesize": 10,
})

C0 = 2.99792458e8
ETA0 = 376.730313
CE, CH, CP, CK = "#1f77b4", "#d62728", "#7f4fbf", "#111111"   # E, H, power, structure
SL = {"style": {"description_width": "104px"},
      "layout": widgets.Layout(width="320px"), "continuous_update": False}

WR = {"WR-284 (S)": (72.14, 34.04), "WR-137 (C)": (34.85, 15.80),
      "WR-90 (X)": (22.86, 10.16), "WR-62 (Ku)": (15.80, 7.90),
      "WR-28 (Ka)": (7.11, 3.56)}


def clean3d(ax, labels=("z", "x", "y")):
    ax.grid(False)
    ax.tick_params(labelsize=7, pad=-2)
    for setter, lab in zip((ax.set_xlabel, ax.set_ylabel, ax.set_zlabel), labels):
        setter(lab, labelpad=-4, fontsize=8)


def te_cutoff(m, n, a, b):
    """Cutoff frequency [Hz] of TE/TM_mn in a guide of a x b metres."""
    return 0.5 * C0 * np.hypot(m / a, n / b)


def envelope(sig):
    """Magnitude of the analytic signal."""
    N = len(sig)
    h = np.zeros(N); h[0] = 1.0; h[N // 2] = 1.0; h[1:N // 2] = 2.0
    return np.abs(np.fft.ifft(np.fft.fft(sig) * h))


def guide_beta(f, fc):
    """Propagation constant; negative return means evanescent decay rate."""
    k = 2 * np.pi * f / C0
    kc = 2 * np.pi * fc / C0
    return np.sqrt(k ** 2 - kc ** 2) if f > fc else -np.sqrt(kc ** 2 - k ** 2)


print(f"c = {C0:.4e} m/s   eta0 = {ETA0:.1f} ohm   "
      f"{len(WR)} standard guides available")

c = 2.9979e+08 m/s   eta0 = 376.7 ohm   5 standard guides available


## 1 — The TEM line: guiding without a cutoff

Two conductors can support a wave whose $E$ and $H$ both lie entirely in the cross-section. The transverse pattern is just the *electrostatic* field of the cross-section, scaled up and down as the wave passes, which is why it works at every frequency including DC. Per unit length the line stores $L$ and $C$, and the telegrapher's equations

$$\frac{\partial v}{\partial z}=-L\frac{\partial i}{\partial t},\qquad
\frac{\partial i}{\partial z}=-C\frac{\partial v}{\partial t}
\quad\Longrightarrow\quad
v_p=\frac{1}{\sqrt{LC}}=\frac{c}{\sqrt{\varepsilon_r}},\qquad Z_0=\sqrt{\frac{L}{C}}$$

give a non-dispersive wave: every frequency travels at the same speed, so a pulse keeps its shape. For coax, $Z_0=\frac{\eta_0}{2\pi\sqrt{\varepsilon_r}}\ln(b/a)$ — note it depends only on the *ratio* of the radii, so a cable can be scaled to any size at constant impedance.

$Z_0$ is not a resistance you can measure with an ohmmeter; it is the fixed ratio $v/i$ that a single travelling wave must obey. The forward wave carries $+Z_0$, the backward wave $-Z_0$, which is the whole reason reflections behave as they do in section 2.

In [2]:
def draw_tem(ba, eps, t_ns, direction):
    Z0 = ETA0 / (2 * np.pi * np.sqrt(eps)) * np.log(ba)
    vp = C0 / np.sqrt(eps)

    fig = plt.figure(figsize=(12.4, 3.9))
    gs = fig.add_gridspec(1, 3, width_ratios=[1, 1.35, 1.1], wspace=0.32, left=0.055, right=0.975, top=0.80, bottom=0.155)

    a0 = fig.add_subplot(gs[0])
    th = np.linspace(0, 2 * np.pi, 300)
    a0.fill(np.cos(th), np.sin(th), color="0.75")
    a0.plot(ba * np.cos(th), ba * np.sin(th), color=CK, lw=3)
    for ang in np.linspace(0, 2 * np.pi, 16, endpoint=False):
        a0.annotate("", xy=(ba * 0.96 * np.cos(ang), ba * 0.96 * np.sin(ang)),
                    xytext=(1.04 * np.cos(ang), 1.04 * np.sin(ang)),
                    arrowprops=dict(arrowstyle="-|>", color=CE, lw=1.1))
    for rr in (1 + (ba - 1) * 0.35, 1 + (ba - 1) * 0.75):
        a0.plot(rr * np.cos(th), rr * np.sin(th), color=CH, lw=1.0, ls="--")
        a0.annotate("", xy=(rr * np.cos(0.14), rr * np.sin(0.14)), xytext=(rr, 0),
                    arrowprops=dict(arrowstyle="-|>", color=CH, lw=1.4))
    a0.set_xlim(-ba * 1.15, ba * 1.15); a0.set_ylim(-ba * 1.15, ba * 1.15)
    a0.set_aspect("equal"); a0.axis("off")
    a0.set_title(f"cross-section: E radial, H azimuthal\nb/a = {ba:.1f}")

    a1 = fig.add_subplot(gs[1])
    z = np.linspace(0, 3.0, 900)
    t = t_ns * 1e-9
    s = 1.0 if direction == "forward (+z)" else -1.0
    zc = (0.4 + s * vp * t / 1.0) % 3.0 if s > 0 else (2.6 + s * vp * t / 1.0) % 3.0
    v = np.exp(-((z - zc) / 0.16) ** 2) * np.cos(2 * np.pi * (z - zc) / 0.36)
    i = v / Z0 * s
    a1.plot(z, v, color=CE, lw=1.7, label="v(z)  [V]")
    a1.set_xlabel("z [m]"); a1.set_ylabel("voltage", color=CE)
    a1.tick_params(axis="y", colors=CE); a1.set_ylim(-1.25, 1.25)
    a1b = a1.twinx()
    a1b.plot(z, i * 1e3, color=CH, lw=1.4, ls="--", label="i(z)  [mA]")
    a1b.set_ylabel("current [mA]", color=CH); a1b.tick_params(axis="y", colors=CH)
    a1b.set_ylim(-1.25 / Z0 * 1e3, 1.25 / Z0 * 1e3)
    a1b.grid(False); a1b.spines["right"].set_visible(True)
    a1b.spines["right"].set_color(CH)
    a1.set_title(f"travelling pulse — v/i = {s * Z0:+.1f} Ω everywhere\n"
                 f"$v_p$ = {vp / 1e8:.2f}×10⁸ m/s = {100 / np.sqrt(eps):.0f}% of c")

    a2 = fig.add_subplot(gs[2])
    r = np.linspace(1.2, 12, 300)
    for e, col in [(1.0, "0.7"), (2.1, CP), (2.25, CE)]:
        a2.plot(r, ETA0 / (2 * np.pi * np.sqrt(e)) * np.log(r), color=col, lw=1.4,
                label=f"$\\varepsilon_r$ = {e}")
    a2.plot(r, ETA0 / (2 * np.pi * np.sqrt(eps)) * np.log(r), color=CK, lw=2.0,
            label=f"current ({eps:.2f})")
    a2.plot([ba], [Z0], "o", color=CH, ms=8, zorder=5)
    for zz, lab in [(50, "50 Ω"), (75, "75 Ω")]:
        a2.axhline(zz, color="0.6", lw=0.8, ls=":")
        a2.text(11.6, zz + 2, lab, fontsize=7, ha="right", color="0.4")
    a2.set_xlabel("b/a"); a2.set_ylabel("$Z_0$ [Ω]"); a2.set_ylim(0, 160)
    a2.legend(fontsize=7); a2.set_title(f"$Z_0$ = {Z0:.1f} Ω")
    plt.show()


w1 = dict(ba=widgets.FloatSlider(value=3.5, min=1.2, max=12.0, step=0.1,
                                 description="b/a ratio:", **SL),
          eps=widgets.FloatSlider(value=2.25, min=1.0, max=5.0, step=0.05,
                                  description="ε_r dielectric:", **SL),
          t_ns=widgets.FloatSlider(value=0, min=0, max=15, step=0.5,
                                   description="time [ns]:", **SL),
          direction=widgets.Dropdown(options=["forward (+z)", "backward (−z)"],
                                     value="forward (+z)", description="wave:",
                                     style={"description_width": "104px"},
                                     layout=widgets.Layout(width="320px")))
display(widgets.VBox([widgets.HBox([w1["ba"], w1["eps"]]),
                      widgets.HBox([w1["t_ns"], w1["direction"]])]),
        widgets.interactive_output(draw_tem, w1))

Output()

## 2 — Mismatch: standing waves and the Smith chart

A load that does not present $Z_0$ cannot satisfy $v/i=Z_0$ with a forward wave alone, so a backward wave appears carrying $v/i=-Z_0$. Their ratio at the load is

$$\Gamma_L=\frac{Z_L-Z_0}{Z_L+Z_0},\qquad
\Gamma(l)=\Gamma_Le^{-2\gamma l},\qquad
\text{VSWR}=\frac{1+|\Gamma|}{1-|\Gamma|}$$

Moving a distance $l$ back toward the generator rotates $\Gamma$ clockwise by $2\beta l$ — twice the electrical length, because the wave makes the trip twice. That rotation is exactly what the Smith chart plots: the complex $\Gamma$ plane with the $Z$ grid painted onto it. One full turn takes $l=\lambda/2$, which is why the standing-wave pattern repeats every half wavelength and why impedance matching is a periodic problem.

| load | $\Gamma_L$ | standing wave |
|---|---|---|
| $Z_L=Z_0$ | $0$ | flat, VSWR $=1$ |
| open | $+1$ | voltage max at the load |
| short | $-1$ | voltage null at the load |
| pure reactance | $\|\Gamma\|=1$ | full standing wave, no power delivered |

In [3]:
def smith_grid(ax):
    t = np.linspace(0, 2 * np.pi, 400)
    for r in (0.0, 0.2, 0.5, 1.0, 2.0, 5.0):
        c, rad = r / (1 + r), 1 / (1 + r)
        ax.plot(c + rad * np.cos(t), rad * np.sin(t), color="0.82", lw=0.7,
                zorder=1)
    for x in (0.2, 0.5, 1.0, 2.0, 5.0):
        for sg in (1, -1):
            X = 1 + (1 / x) * np.cos(t)
            Y = sg / x + (1 / x) * np.sin(t)
            keep = X ** 2 + Y ** 2 <= 1.0
            ax.plot(np.where(keep, X, np.nan), np.where(keep, Y, np.nan),
                    color="0.82", lw=0.7, zorder=1)
    ax.plot(np.cos(t), np.sin(t), color=CK, lw=1.2, zorder=2)
    ax.plot([-1, 1], [0, 0], color="0.82", lw=0.7, zorder=1)


def draw_mismatch(RL, XL, length, loss_db):
    ZL = complex(RL, XL)
    GL = (ZL - 1) / (ZL + 1) if abs(ZL + 1) > 1e-9 else 1.0 + 0j
    mag = abs(GL)
    vswr = (1 + mag) / (1 - mag) if mag < 0.999 else np.inf
    alpha = loss_db / 8.686

    fig = plt.figure(figsize=(12.2, 4.4))
    gs = fig.add_gridspec(1, 2, width_ratios=[1.45, 1], wspace=0.28, left=0.055, right=0.975, top=0.80, bottom=0.155)

    a0 = fig.add_subplot(gs[0])
    l = np.linspace(0, length, 1200)
    G = GL * np.exp(-2 * alpha * l) * np.exp(-4j * np.pi * l)
    V = np.abs(1 + G)
    I = np.abs(1 - G)
    a0.plot(-l, V, color=CE, lw=1.7, label="|V| / |V⁺|")
    a0.plot(-l, I, color=CH, lw=1.4, ls="--", label="|I|·Z₀ / |V⁺|")
    a0.axvline(0, color=CK, lw=2)
    a0.text(0.02, 0.06, "load", transform=a0.transAxes, fontsize=8)
    for kk in range(1, int(2 * length) + 2):     # minima every half wavelength
        zmin = -(np.angle(GL) + np.pi + 2 * np.pi * (kk - 1)) / (4 * np.pi)
        if -length < zmin <= 0 or -length < -abs(zmin) <= 0:
            pass
    a0.axhline(1 + mag, color="0.6", lw=0.8, ls=":")
    a0.axhline(1 - mag, color="0.6", lw=0.8, ls=":")
    a0.set_xlim(-length, 0.04 * length); a0.set_ylim(0, 2.15)
    a0.set_xlabel("distance from load  [wavelengths]")
    a0.set_ylabel("normalized magnitude")
    a0.legend(fontsize=8, loc="upper left")
    rl = -20 * np.log10(mag) if mag > 1e-6 else np.inf
    a0.set_title(f"VSWR = {vswr:.2f}   |Γ| = {mag:.3f}   "
                 f"return loss = {rl:.1f} dB")

    a1 = fig.add_subplot(gs[1])
    smith_grid(a1)
    t = np.linspace(0, 2 * np.pi, 300)
    a1.plot(mag * np.cos(t), mag * np.sin(t), color="0.55", lw=1.0, ls=":")
    Gt = GL * np.exp(-2 * alpha * l) * np.exp(-4j * np.pi * l)
    a1.plot(Gt.real, Gt.imag, color=CP, lw=1.8, zorder=3)
    a1.plot([GL.real], [GL.imag], "o", color=CH, ms=9, zorder=4)
    a1.plot([Gt[-1].real], [Gt[-1].imag], "s", color=CE, ms=8, zorder=4)
    Zin = (1 + Gt[-1]) / (1 - Gt[-1]) if abs(1 - Gt[-1]) > 1e-9 else complex(1e3, 0)
    a1.set_xlim(-1.1, 1.1); a1.set_ylim(-1.1, 1.1)
    a1.set_aspect("equal"); a1.axis("off")
    a1.set_title(f"Γ plane — ● load, ■ input after {length:.2f}λ\n"
                 f"$z_L$ = {RL:.2f}{XL:+.2f}j → $z_{{in}}$ = "
                 f"{Zin.real:.2f}{Zin.imag:+.2f}j")
    plt.show()


w2 = dict(RL=widgets.FloatSlider(value=0.4, min=0.0, max=5.0, step=0.1,
                                 description="R_L / Z₀:", **SL),
          XL=widgets.FloatSlider(value=0.8, min=-5.0, max=5.0, step=0.1,
                                 description="X_L / Z₀:", **SL),
          length=widgets.FloatSlider(value=0.5, min=0.05, max=2.0, step=0.05,
                                     description="line length [λ]:", **SL),
          loss_db=widgets.FloatSlider(value=0.0, min=0.0, max=3.0, step=0.1,
                                      description="loss [dB/λ]:", **SL))
display(widgets.VBox([widgets.HBox([w2["RL"], w2["XL"]]),
                      widgets.HBox([w2["length"], w2["loss_db"]])]),
        widgets.interactive_output(draw_mismatch, w2))

Output()

## 3 — Remove a conductor: cutoff appears

A hollow pipe has only one conductor, so no TEM solution exists — there is nowhere for the electrostatic field lines to terminate. What survives are patterns that fit an integer number of half-cycles across the cross-section:

$$f_{c,mn}=\frac{c}{2}\sqrt{\left(\frac{m}{a}\right)^2+\left(\frac{n}{b}\right)^2},\qquad
\beta=\frac{2\pi}{c}\sqrt{f^2-f_c^2}$$

Below $f_c$ the square root turns imaginary: the field still exists but decays as $e^{-\alpha z}$ without transporting power. This is not resistive loss — nothing is dissipated, the guide simply reflects everything back. A waveguide is a high-pass filter made of geometry.

Choosing $b\le a/2$ puts TE$_{20}$ at exactly $2f_{c10}$ and gives the widest single-mode band, which is why every standard rectangular guide has roughly a 2:1 aspect ratio and is used over about $1.25f_{c}$ to $1.9f_{c}$.

In [ ]:
def mode_list(a, b, nmax=4):
    out = []
    for m in range(nmax + 1):
        for n in range(nmax + 1):
            if m == 0 and n == 0:
                continue
            out.append((f"TE{m}{n}", te_cutoff(m, n, a, b)))
            if m > 0 and n > 0:
                out.append((f"TM{m}{n}", te_cutoff(m, n, a, b)))
    return sorted(out, key=lambda p: p[1])


def draw_cutoff(preset, a_mm, b_mm, f_GHz):
    if preset != "custom":
        a_mm, b_mm = WR[preset]
    a, b, f = a_mm * 1e-3, b_mm * 1e-3, f_GHz * 1e9
    modes = mode_list(a, b)
    fc10 = te_cutoff(1, 0, a, b)
    fc_next = min(fc for nm, fc in modes if fc > fc10 * 1.0001)

    fig = plt.figure(figsize=(12.2, 4.4))
    gs = fig.add_gridspec(1, 2, width_ratios=[1, 1.15], wspace=0.26, left=0.055, right=0.975, top=0.80, bottom=0.155)

    a0 = fig.add_subplot(gs[0])
    a0.axvspan(fc10 / 1e9, fc_next / 1e9, color=CP, alpha=0.13)
    show = [p for p in modes if p[1] < 4.2 * fc10][:11]
    for i, (nm, fc) in enumerate(show):
        col = CE if f > fc else "0.65"
        a0.plot([fc / 1e9, fc / 1e9], [i - 0.34, i + 0.34], color=col, lw=3)
        a0.text(fc / 1e9 * 1.02, i, f"{nm}  {fc / 1e9:.2f} GHz",
                fontsize=7.5, va="center", color=col)
    a0.axvline(f_GHz, color=CH, lw=1.8)
    a0.text(f_GHz, len(show) - 0.3, f" f = {f_GHz:.2f} GHz", color=CH, fontsize=8)
    a0.set_ylim(-0.8, len(show) + 0.2); a0.set_yticks([])
    a0.set_xlim(0, 4.2 * fc10 / 1e9)
    a0.set_xlabel("frequency [GHz]")
    n_prop = sum(1 for nm, fc in modes if f > fc)
    a0.set_title(f"{preset}  {a_mm:.2f}×{b_mm:.2f} mm — "
                 f"{n_prop} mode(s) propagating\nshaded: single-mode band "
                 f"{fc10 / 1e9:.2f}–{fc_next / 1e9:.2f} GHz")

    a1 = fig.add_subplot(gs[1])
    fg = np.linspace(0.05, 4.2 * fc10, 700)
    a1.plot(2 * np.pi * fg / C0 / 100, fg / 1e9, color="0.6", lw=1.0, ls="--",
            label="light line  β = k")
    for nm, fc in show[:5]:
        bb = np.where(fg > fc, np.sqrt(np.maximum((2 * np.pi * fg / C0) ** 2
                                                  - (2 * np.pi * fc / C0) ** 2, 0)),
                      np.nan)
        a1.plot(bb / 100, fg / 1e9, lw=1.5, label=nm)
    a1.axhline(f_GHz, color=CH, lw=1.2, ls=":")
    a1.set_xlabel("β [rad/cm]"); a1.set_ylabel("frequency [GHz]")
    a1.set_xlim(0, 2 * np.pi * 4.2 * fc10 / C0 / 100)
    a1.set_ylim(0, 4.2 * fc10 / 1e9)
    a1.legend(fontsize=7); a1.set_title("dispersion: every mode hugs the light\n"
                                        "line at high f, bends to β→0 at cutoff")
    plt.show()


w3 = dict(preset=widgets.Dropdown(options=["custom"] + list(WR),
                                  value="WR-90 (X)", description="standard guide:",
                                  style={"description_width": "104px"},
                                  layout=widgets.Layout(width="320px")),
          a_mm=widgets.FloatSlider(value=22.86, min=5.0, max=80.0, step=0.5,
                                   description="a [mm]:", **SL),
          b_mm=widgets.FloatSlider(value=10.16, min=2.0, max=40.0, step=0.5,
                                   description="b [mm]:", **SL),
          f_GHz=widgets.FloatSlider(value=10.0, min=0.5, max=40.0, step=0.1,
                                    description="frequency [GHz]:", **SL))
display(widgets.VBox([widgets.HBox([w3["preset"], w3["f_GHz"]]),
                      widgets.HBox([w3["a_mm"], w3["b_mm"]])]),
        widgets.interactive_output(draw_cutoff, w3))

Output()

## 4 — What a mode actually looks like, and where it puts its currents

For TE$_{mn}$ the longitudinal magnetic field sets everything else. With $k_c^2=(m\pi/a)^2+(n\pi/b)^2$,

$$H_z=H_0\cos\frac{m\pi x}{a}\cos\frac{n\pi y}{b}e^{-j\beta z},\qquad
E_t=\frac{-j\omega\mu}{k_c^2}\,\hat{z}\times\nabla_tH_z,\qquad
H_t=\frac{-j\beta}{k_c^2}\nabla_tH_z$$

The mode terminates on the walls as a surface current $\mathbf{J}_s=\hat{n}\times\mathbf{H}$, and that current is the bridge back to `antenna_propagation.ipynb`: a slot cut through the wall radiates only if it *interrupts* current. On the broad wall of a TE$_{10}$ guide the current runs purely longitudinally along the centreline, so a longitudinal slot cut there is invisible — it slips between the current lines. Offset that same slot toward the sidewall, or rotate it transverse, and it cuts current and radiates. Slotted-waveguide arrays are built by choosing each slot's offset to set its coupling, which is why the third panel is a antenna design chart as much as a field plot.

In [ ]:
def te_fields(m, n, a, b, beta, X, Y, phase):
    kx, ky = m * np.pi / a, n * np.pi / b
    kc2 = kx ** 2 + ky ** 2
    cx, sx = np.cos(kx * X), np.sin(kx * X)
    cy, sy = np.cos(ky * Y), np.sin(ky * Y)
    s, c = np.sin(phase), np.cos(phase)
    Ex = (ky / kc2) * cx * sy * (-s)
    Ey = -(kx / kc2) * sx * cy * (-s)
    Hx = (beta * kx / kc2) * sx * cy * (-s)
    Hy = (beta * ky / kc2) * cx * sy * (-s)
    Hz = cx * cy * c
    return Ex, Ey, Hx, Hy, Hz


def draw_mode(m, n, preset, fratio, wt_deg):
    if m == 0 and n == 0:
        fig = plt.figure(figsize=(7, 1.5))
        fig.text(0.5, 0.5, "TE$_{00}$ does not exist — set m or n to at least 1",
                 ha="center", va="center", fontsize=12, color=CH)
        plt.show(); return
    a_mm, b_mm = WR[preset]
    a, b = a_mm * 1e-3, b_mm * 1e-3
    fc = te_cutoff(m, n, a, b)
    f = fratio * fc
    beta = guide_beta(f, fc)
    prop = beta > 0
    lg = 2 * np.pi / beta if prop else 4 * a
    wt = np.deg2rad(wt_deg)

    fig = plt.figure(figsize=(12.6, 4.3))
    gs = fig.add_gridspec(1, 3, width_ratios=[1, 1.35, 1.15], wspace=0.3, left=0.055, right=0.975, top=0.80, bottom=0.155)

    a0 = fig.add_subplot(gs[0])
    xs = np.linspace(0, a, 90); ys = np.linspace(0, b, 45)
    X, Y = np.meshgrid(xs, ys, indexing="ij")
    Ex, Ey, Hx, Hy, Hz = te_fields(m, n, a, b, max(beta, 1.0), X, Y, wt)
    a0.contourf(X * 1e3, Y * 1e3, np.hypot(Ex, Ey), levels=18, cmap="Blues")
    q = slice(None, None, 7), slice(None, None, 4)
    a0.quiver(X[q] * 1e3, Y[q] * 1e3, Ex[q], Ey[q], color=CE, scale_units="xy",
              scale=np.nanmax(np.hypot(Ex, Ey)) / (0.12 * a_mm) + 1e-9, width=0.008)
    a0.add_patch(plt.Rectangle((0, 0), a_mm, b_mm, fill=False, ec=CK, lw=2.5))
    a0.set_xlim(-1, a_mm + 1); a0.set_ylim(-1, b_mm + 1)
    a0.set_aspect("equal"); a0.grid(False)
    a0.set_xlabel("x [mm]"); a0.set_ylabel("y [mm]")
    a0.set_title(f"transverse E — TE$_{{{m}{n}}}$")

    a1 = fig.add_subplot(gs[1], projection="3d")
    xs2 = np.linspace(0.08 * a, 0.92 * a, 6)
    ys2 = np.linspace(0.15 * b, 0.85 * b, 3)
    zs2 = np.linspace(0, 1.4 * lg, 13)
    X2, Y2, Z2 = np.meshgrid(xs2, ys2, zs2, indexing="ij")
    ph = wt - (beta if prop else 0) * Z2
    env = np.exp(-abs(beta) * Z2) if not prop else 1.0
    E2x, E2y, *_ = te_fields(m, n, a, b, max(beta, 1.0), X2, Y2, ph)
    E2x, E2y = E2x * env, E2y * env
    mag = np.hypot(E2x, E2y)
    sc = 0.42 * b / max(mag.max(), 1e-12)
    a1.quiver(Z2 * 1e3, X2 * 1e3, Y2 * 1e3, np.zeros_like(E2x),
              E2x * sc * 1e3, E2y * sc * 1e3, color=CE, lw=0.9,
              arrow_length_ratio=0.32)
    for yy in (0, b_mm):
        a1.plot([0, zs2[-1] * 1e3, zs2[-1] * 1e3, 0, 0],
                [0, 0, a_mm, a_mm, 0], [yy] * 5, color="0.6", lw=0.8)
    a1.set_xlim(0, zs2[-1] * 1e3); a1.set_ylim(0, a_mm); a1.set_zlim(0, b_mm)
    a1.set_box_aspect((2.7, 1.0, b_mm / a_mm))
    clean3d(a1, ("z [mm]", "x [mm]", "y [mm]"))
    a1.view_init(elev=20, azim=-66)
    state = f"λ_g = {lg * 1e3:.1f} mm" if prop else "EVANESCENT"
    a1.set_title(f"f = {f / 1e9:.2f} GHz = {fratio:.2f}·f_c   {state}")

    a2 = fig.add_subplot(gs[2])
    Xw, Zw = np.meshgrid(np.linspace(0, a, 70), np.linspace(0, 1.2 * lg, 70),
                         indexing="ij")
    phw = wt - (beta if prop else 0) * Zw
    _, _, Hxw, _, Hzw = te_fields(m, n, a, b, max(beta, 1.0), Xw,
                                  np.full_like(Xw, b), phw)
    Jz, Jx = Hxw, -Hzw                      # J = n x H on the y = b wall
    a2.contourf(Zw * 1e3, Xw * 1e3, np.hypot(Jx, Jz), levels=18, cmap="Reds")
    qq = slice(None, None, 6), slice(None, None, 6)
    a2.quiver(Zw[qq] * 1e3, Xw[qq] * 1e3, Jz[qq], Jx[qq], color=CH,
              width=0.006, scale=np.nanmax(np.hypot(Jx, Jz)) * 18 + 1e-9)
    if (m, n) == (1, 0):
        a2.add_patch(plt.Rectangle((0.30 * lg * 1e3, a_mm / 2 - 0.9),
                                   0.16 * lg * 1e3, 1.8, fill=False,
                                   ec="k", lw=1.6))
        a2.text(0.38 * lg * 1e3, a_mm / 2 + 2.4, "centred: no radiation",
                fontsize=7, ha="center")
        a2.add_patch(plt.Rectangle((0.70 * lg * 1e3, 0.24 * a_mm - 0.9),
                                   0.16 * lg * 1e3, 1.8, fill=False,
                                   ec="w", lw=1.6))
        a2.text(0.78 * lg * 1e3, 0.24 * a_mm + 2.4, "offset: radiates",
                fontsize=7, ha="center", color="w")
    a2.set_xlabel("z [mm]"); a2.set_ylabel("x [mm]"); a2.grid(False)
    a2.set_title("wall current on the broad wall\n$J_s=\\hat{n}\\times H$")
    plt.show()


w4 = dict(m=widgets.IntSlider(value=1, min=0, max=3, step=1, description="index m:", **SL),
          n=widgets.IntSlider(value=0, min=0, max=3, step=1, description="index n:", **SL),
          preset=widgets.Dropdown(options=list(WR), value="WR-90 (X)",
                                  description="guide:",
                                  style={"description_width": "104px"},
                                  layout=widgets.Layout(width="320px")),
          fratio=widgets.FloatSlider(value=1.5, min=0.7, max=3.0, step=0.05,
                                     description="f / f_c of mode:", **SL),
          wt_deg=widgets.FloatSlider(value=0, min=0, max=350, step=10,
                                     description="ωt [deg]:", **SL))
display(widgets.VBox([widgets.HBox([w4["m"], w4["n"], w4["preset"]]),
                      widgets.HBox([w4["fratio"], w4["wt_deg"]])]),
        widgets.interactive_output(draw_mode, w4))

Output()

## 5 — The price of cutoff: dispersion

The TEM line delivered every frequency at the same speed. A guide cannot, because $\beta$ depends on $f$ non-linearly:

$$v_p=\frac{\omega}{\beta}=\frac{c}{\sqrt{1-(f_c/f)^2}}>c,\qquad
v_g=\frac{d\omega}{d\beta}=c\sqrt{1-(f_c/f)^2}<c,\qquad v_pv_g=c^2$$

On the $\omega$–$\beta$ diagram $v_p$ is the slope of the **chord** from the origin and $v_g$ the slope of the **tangent** — both drawn on the left panel, which is the fastest way to see why one exceeds $c$ while the other does not. Nothing outruns light: $v_p$ describes the phase pattern sliding along the wall, which carries no information.

A pulse is a band of frequencies, each with its own $v_g$, so it spreads. Push the carrier toward cutoff and the tangent flattens: the pulse arrives late and smeared. Push far enough and the description collapses altogether — once the lower edge of the band crosses $f_c$ that part of the spectrum is reflected rather than delayed, the pulse is filtered as well as dispersed, and the single number $L/v_g$ stops predicting when anything arrives. The panel says so when it happens. This is why guides run well above $f_c$, and it is the same mechanism that limits bit rates in the fibre of section 6.

In [6]:
def draw_dispersion(fratio, bw_pct, L_m, preset):
    a_mm, b_mm = WR[preset]
    fc = te_cutoff(1, 0, a_mm * 1e-3, b_mm * 1e-3)
    f0 = fratio * fc
    vg0 = C0 * np.sqrt(max(1 - (fc / f0) ** 2, 1e-9))
    vp0 = C0 ** 2 / vg0

    fig = plt.figure(figsize=(12.2, 4.4))
    gs = fig.add_gridspec(1, 2, width_ratios=[1, 1.3], wspace=0.28, left=0.055, right=0.975, top=0.80, bottom=0.155)

    a0 = fig.add_subplot(gs[0])
    fg = np.linspace(fc * 1.0001, fc * 4, 600)
    bg = 2 * np.pi * np.sqrt(fg ** 2 - fc ** 2) / C0
    a0.plot(bg / 100, fg / 1e9, color=CE, lw=1.8, label="TE₁₀")
    a0.plot(2 * np.pi * fg / C0 / 100, fg / 1e9, color="0.6", lw=1.0, ls="--",
            label="light line")
    b0 = 2 * np.pi * np.sqrt(f0 ** 2 - fc ** 2) / C0
    a0.plot([0, b0 / 100 * 1.6], [0, f0 / 1e9 * 1.6 * 1.0], color=CH, lw=1.2,
            ls=":", label=f"chord → $v_p$ = {vp0 / C0:.2f}c")
    tang = np.linspace(b0 * 0.4, b0 * 1.7, 10)
    a0.plot(tang / 100, (f0 + vg0 * (tang - b0) / (2 * np.pi)) / 1e9,
            color=CP, lw=1.6, label=f"tangent → $v_g$ = {vg0 / C0:.2f}c")
    a0.plot([b0 / 100], [f0 / 1e9], "o", color=CK, ms=7, zorder=5)
    a0.set_xlim(0, bg.max() / 100); a0.set_ylim(0, fg.max() / 1e9)
    a0.set_xlabel("β [rad/cm]"); a0.set_ylabel("f [GHz]")
    a0.legend(fontsize=7, loc="upper left")
    a0.set_title(f"chord vs tangent at f = {f0 / 1e9:.2f} GHz")

    N = 4096
    fs = 40 * fc
    t = (np.arange(N) - N // 2) / fs
    bw = bw_pct / 100 * f0
    sigma = 1 / (2 * np.pi * bw)
    x = np.exp(-0.5 * (t / sigma) ** 2) * np.cos(2 * np.pi * f0 * t)
    X = np.fft.rfft(np.fft.ifftshift(x))
    fr = np.fft.rfftfreq(N, 1 / fs)
    kz = np.where(fr > fc, 2 * np.pi * np.sqrt(np.maximum(fr ** 2 - fc ** 2, 0)) / C0,
                  0.0)
    att = np.where(fr > fc, 1.0,
                   np.exp(-2 * np.pi * np.sqrt(np.maximum(fc ** 2 - fr ** 2, 0))
                          / C0 * L_m))
    y = np.fft.fftshift(np.fft.irfft(X * np.exp(-1j * kz * L_m) * att, N))
    P = np.abs(X) ** 2
    below = np.trapezoid(P[fr <= fc], fr[fr <= fc]) / np.trapezoid(P, fr)

    tp = (t + L_m / vg0) * 1e9
    a1 = fig.add_subplot(gs[1])
    a1.plot(t * 1e9, x, color="0.75", lw=0.7)
    a1.plot(t * 1e9, envelope(x), color=CK, lw=1.5, label="input envelope")
    a1.plot(tp, y / max(np.abs(y).max(), 1e-12), color=CE, lw=0.7, alpha=0.6)
    ey = envelope(y)
    a1.plot(tp, ey / max(ey.max(), 1e-12), color=CE, lw=1.7,
            label=f"after {L_m:.2f} m (shifted by $L/v_g$)")
    w_in = np.trapezoid(envelope(x), t) / max(envelope(x).max(), 1e-12)
    w_out = np.trapezoid(ey, t) / max(ey.max(), 1e-12)
    a1.set_xlim(-6 * sigma * 1e9, 6 * sigma * 1e9)
    a1.set_xlabel("time [ns]"); a1.set_ylabel("amplitude")
    a1.legend(fontsize=7, loc="upper right")
    a1.set_title(f"group delay = {L_m / vg0 * 1e9:.2f} ns   "
                 f"broadening ×{w_out / max(w_in, 1e-12):.2f}   "
                 f"bandwidth {bw / 1e9:.2f} GHz")
    if below > 0.01:
        a1.text(0.02, 0.04, f"{below:.0%} of the pulse energy sits below $f_c$:\n"
                            "that part is reflected away, not delayed —\n"
                            "the single number $L/v_g$ no longer describes arrival",
                transform=a1.transAxes, fontsize=7.5, color=CH,
                bbox=dict(boxstyle="round", fc="w", ec=CH, alpha=0.9))
    plt.show()


w5 = dict(fratio=widgets.FloatSlider(value=1.5, min=1.02, max=3.0, step=0.02,
                                     description="f₀ / f_c:", **SL),
          bw_pct=widgets.FloatSlider(value=8, min=1, max=30, step=1,
                                     description="bandwidth [%]:", **SL),
          L_m=widgets.FloatSlider(value=1.0, min=0.05, max=5.0, step=0.05,
                                  description="guide length [m]:", **SL),
          preset=widgets.Dropdown(options=list(WR), value="WR-90 (X)",
                                  description="guide:",
                                  style={"description_width": "104px"},
                                  layout=widgets.Layout(width="320px")))
display(widgets.VBox([widgets.HBox([w5["fratio"], w5["bw_pct"]]),
                      widgets.HBox([w5["L_m"], w5["preset"]])]),
        widgets.interactive_output(draw_dispersion, w5))

Output()

## 6 — Remove the last conductor: trapping by total internal reflection

A dielectric slab or fibre has no metal at all. A ray inside the core striking the boundary above the critical angle is reflected with unit magnitude, so the wave is trapped by refraction alone:

$$\theta_c=\arcsin\frac{n_2}{n_1},\qquad
\text{NA}=\sqrt{n_1^2-n_2^2},\qquad
V=\frac{2\pi d}{\lambda}\text{NA}$$

Requiring the round trip across the core to return in phase quantizes the allowed angles exactly as the metal walls did, giving the eigenvalue equation $\sqrt{V^2-u^2}=u\tan\!\left(u-\frac{m\pi}{2}\right)$ with $u=\kappa d$, solved numerically below. The count of guided modes is $\lceil 2V/\pi\rceil$, so a slab is single-mode for $V<\pi/2$ and a step-index fibre for $V<2.405$.

The difference from a metal guide is the tail: a dielectric mode is not confined by a wall, so its field leaks into the cladding as $e^{-\gamma|x|}$. The nearer a mode is to its own cutoff, the further out it reaches — which is how fibre couplers work, and why bending a fibre makes it lose light.

In [7]:
def slab_modes(V, mmax=12):
    """Solve sqrt(V^2-u^2) = u*tan(u - m*pi/2) branch by branch."""
    out = []
    for m in range(mmax):
        lo, hi = m * np.pi / 2, min((m + 1) * np.pi / 2, V)
        if hi - lo < 1e-6:
            break
        lo, hi = lo + 1e-9, hi - 1e-9

        def g(u):
            return u * np.tan(u - m * np.pi / 2) - np.sqrt(max(V ** 2 - u ** 2, 0))

        if g(lo) * g(hi) > 0:
            continue
        out.append((m, brentq(g, lo, hi, xtol=1e-12)))
    return out


def draw_slab(n1, n2, d_over_lam):
    n2 = min(n2, n1 - 0.001)
    NA = np.sqrt(n1 ** 2 - n2 ** 2)
    thc = np.degrees(np.arcsin(n2 / n1))
    V = 2 * np.pi * d_over_lam * NA
    modes = slab_modes(V)

    fig = plt.figure(figsize=(12.4, 4.3))
    gs = fig.add_gridspec(1, 3, width_ratios=[1.25, 1, 1], wspace=0.3, left=0.055, right=0.975, top=0.80, bottom=0.155)

    a0 = fig.add_subplot(gs[0])
    a0.axhspan(-1, 1, color=CE, alpha=0.16)
    a0.axhline(1, color=CK, lw=1.6); a0.axhline(-1, color=CK, lw=1.6)
    for ang, col in [(thc + (90 - thc) * 0.55, CP), (thc + (90 - thc) * 0.15, CP),
                     (thc * 0.75, CH)]:
        guided = ang > thc
        x, y, vy = [0.0], [0.0], 1.0
        slope = 1 / np.tan(np.radians(ang))
        while x[-1] < 6 and (guided or len(x) < 3):
            dx = (1 - y[-1] * vy) / max(abs(slope), 1e-6) if vy > 0 else \
                 (1 + y[-1]) / max(abs(slope), 1e-6)
            nx = x[-1] + dx
            x.append(nx); y.append(vy); vy = -vy
        a0.plot(x, y, color=col, lw=1.5,
                label=f"θ = {ang:.0f}°  " + ("guided" if guided else "escapes"))
        if not guided:
            a0.annotate("", xy=(x[1] + 1.2, 1 + 1.2 * 0.6), xytext=(x[1], 1),
                        arrowprops=dict(arrowstyle="-|>", color=CH, lw=1.5))
    a0.set_xlim(0, 6); a0.set_ylim(-2.2, 2.2)
    a0.set_yticks([-1, 0, 1]); a0.set_yticklabels(["−d", "0", "+d"])
    a0.set_xlabel("propagation direction"); a0.grid(False)
    a0.set_title(f"$n_1$={n1:.3f}, $n_2$={n2:.3f}   θ_c = {thc:.1f}°\n"
                 f"NA = {NA:.3f}")

    a1 = fig.add_subplot(gs[1])
    xx = np.linspace(-3, 3, 800)
    for k, (m, u) in enumerate(modes):
        g = np.sqrt(max(V ** 2 - u ** 2, 0))
        decay = np.exp(-g * (np.abs(xx) - 1.0))
        if m % 2 == 0:
            core, tail = np.cos(u * xx), np.cos(u) * decay
        else:
            core, tail = np.sin(u * xx), np.sign(xx) * np.sin(u) * decay
        prof = np.where(np.abs(xx) <= 1, core, tail)
        prof = prof / max(np.abs(prof).max(), 1e-12)
        a1.plot(prof + 2.4 * k, xx, lw=1.5, label=f"m={m}")
        a1.axvline(2.4 * k, color="0.85", lw=0.6)
    a1.axhspan(-1, 1, color=CE, alpha=0.16)
    a1.axhline(1, color=CK, lw=1.0); a1.axhline(-1, color=CK, lw=1.0)
    a1.set_ylim(-3, 3); a1.set_xticks([]); a1.set_ylabel("x / d")
    a1.grid(False); a1.legend(fontsize=7, ncol=2)
    a1.set_title(f"V = {V:.2f} → {len(modes)} guided mode(s)\n"
                 "evanescent tails in the cladding")

    a2 = fig.add_subplot(gs[2])
    Vs = np.linspace(0.05, max(12, V * 1.2), 260)
    for m in range(6):
        neff = []
        for Vv in Vs:
            sol = [u for mm, u in slab_modes(Vv, mmax=m + 1) if mm == m]
            if sol:
                b = 1 - (sol[0] / Vv) ** 2
                neff.append(np.sqrt(n2 ** 2 + b * (n1 ** 2 - n2 ** 2)))
            else:
                neff.append(np.nan)
        a2.plot(Vs, neff, lw=1.4, label=f"m={m}")
    a2.axhline(n1, color=CK, lw=0.9, ls="--")
    a2.axhline(n2, color=CK, lw=0.9, ls="--")
    a2.axvline(V, color=CH, lw=1.4)
    a2.axvline(np.pi / 2, color=CP, lw=1.0, ls=":")
    a2.text(np.pi / 2 * 1.05, n2 + 0.12 * (n1 - n2), "single-mode\nV < π/2",
            fontsize=7, color=CP)
    a2.set_xlabel("V number"); a2.set_ylabel("$n_{eff}$")
    a2.set_ylim(n2 - 0.02 * (n1 - n2 + 0.01), n1 + 0.02 * (n1 - n2 + 0.01))
    a2.legend(fontsize=7, ncol=2)
    a2.set_title("modes switch on one by one\nas the core widens")
    plt.show()


w6 = dict(n1=widgets.FloatSlider(value=1.470, min=1.40, max=2.00, step=0.005,
                                 description="core n₁:", **SL),
          n2=widgets.FloatSlider(value=1.455, min=1.30, max=1.99, step=0.005,
                                 description="cladding n₂:", **SL),
          d_over_lam=widgets.FloatSlider(value=4.0, min=0.2, max=20.0, step=0.2,
                                         description="half-width d/λ:", **SL))
display(widgets.HBox([w6["n1"], w6["n2"], w6["d_over_lam"]]),
        widgets.interactive_output(draw_slab, w6))

Output()